# Cleanup, variable selection, and QC

In this notebook we will go through the QC procedure to select, and clean the NHANES data

## Initial set up

- Import libraries
- Read data
- Select phenotypes and covariates

In [1]:
#### IMPORT MODULES
import modules
import clarite
import os
import numpy as np
import pandas as pd
import warnings

#### SET PATHS
paths = modules.set_project_paths()

#### READ DATA
nhanes              = modules.NhanesData(modules.load_raw_nhanes(paths[1]))
var_description     = modules.load_var_information(paths[1])
var_category        = modules.load_var_information(paths[1], description=False)
weights_discovery   = modules.WeightData(modules.load_weights(paths[1]))
weights_replication = modules.WeightData(modules.load_weights(paths[1], False))

#### CLEAN SURVEY WEIGHTS
print('Cleaning discovery weights \n')
weights_discovery.remove_weights_not_in_data()
print('Cleaning replication weights \n')
weights_replication.remove_weights_not_in_data()
print('')

#### SUMMARY OF NHANES DATASET
clarite.describe.summarize(nhanes.data)

rpy2 ModuleSpec(name='rpy2', loader=<_frozen_importlib_external.SourceFileLoader object at 0x7f838d78e450>, origin='/storage/home/tug156/work/software/anaconda3/envs/py_clarite/lib/python3.7/site-packages/rpy2/__init__.py', submodule_search_locations=['/storage/home/tug156/work/software/anaconda3/envs/py_clarite/lib/python3.7/site-packages/rpy2'])
Cleaning discovery weights 

Cleaning replication weights 

WTSTH2YR weight, is not in data columns
WTSPH2YR weight, is not in data columns
WTSPP2YR weight, is not in data columns
Removing LBXTSH variable
Removing URX24D variable
Removing URX25T variable
Removing URX4FP variable
Removing URXACE variable
Removing URXATZ variable
Removing URXCB3 variable
Removing URXCCC variable
Removing URXCMH variable
Removing URXCPM variable
Removing URXDEE variable
Removing URXDIZ variable
Removing URXDPY variable
Removing URXMET variable
Removing URXOPM variable
Removing URXP08 variable
Removing URXPAR variable
Removing URXTCC variable

41,474 observations

In [2]:
#### SELECTING PHENOTYPES AND COVARIATES
phenotypes = modules.get_phenotypes(True, var_description)        
covariates = modules.get_covariates()
print('')
print(covariates)
print('')
print('There are ' + str(len(phenotypes)) + ' phenotypes')

	URXUCR = Creatinine, urine (mg/dL)
	LBXSCR = Creatinine (mg/dL)
	LBXSATSI = ALT (U/L)
	LBXSAL = Albumin (g/dL)
	URXUMA = Albumin, urine (ug/mL)
	LBXSAPSI = Alkaline phosphotase (U/L)
	LBXSASSI = AST (U/L)
	LBXSC3SI = Bicarbonate (mmol/L)
	LBXSBU = Blood urea nitrogen (mg/dL)
	LBXBAP = Bone alkaline phosphotase (ug/L)
	LBXCPSI = C-peptide: SI(nmol/L)
	LBXCRP = C-reactive protein(mg/dL)
	LBXSCLSI = Chloride (mmol/L)
	LBXSCH = Cholesterol, total (mg/dL)
	LBDHDL = Direct HDL-Cholesterol (mg/dL)
	LBDLDL = LDL-cholesterol (mg/dL)
	LBXFER = Ferritin (ng/mL)
	LBXSGTSI = GGT (U/L)
	LBXSGB = Globulin (g/dL)
	LBXGLU = Glucose, plasma (mg/dL)
	LBXGH = Glycohemoglobin (%)
	LBXHCY = Homocysteine (umol/L)
	LBXSLDSI = LDH (U/L)
	LBXMMA = Methylmalonic acid (umol/L)
	LBXSOSSI = Osmolality (mOsml/L)
	LBXSPH = Phosphorus (mg/dL)
	LBXSKSI = Potassium (mmol/L)
	LBXEPP = Protoporphyrin (ug/dL RBC)
	LBXSNASI = Sodium (mmol/L)
	LBXTIB = TIBC (ug/dL)
	LBXSTB = Bilirubin, total (mg/dL)
	LBXSCA = Calcium, total

## Selecting participants

We'll keep participants older than 18 years, and remove those that have missing data in any of the covariates

In [3]:
#### KEEP ADULTS
nhanes.keep_adults()
clarite.describe.summarize(nhanes.data)

Keeping only adults in the dataset:
-----------------------------------
22,624 observations of 1,190 variables
	0 Binary Variables
	0 Categorical Variables
	1,189 Continuous Variables
	1 Unknown-Type Variables



In [4]:
#### REMOVE PARTICIPANTS WITH MISSING COVARIATES
nhanes.data = clarite.modify.rowfilter_incomplete_obs(nhanes.data, only=covariates)
clarite.describe.summarize(nhanes.data)

Running rowfilter_incomplete_obs
--------------------------------------------------------------------------------
Removed 3,687 of 22,624 observations (16.30%) due to NA values in any of 8 variables
18,937 observations of 1,190 variables
	0 Binary Variables
	0 Categorical Variables
	1,189 Continuous Variables
	1 Unknown-Type Variables



## Variable selection and clenup

Here we will:
- Drop variables that don't have 4 year weights
- Drop any variables that are indeterminant according to the NHANES data dictionary
- Drop any non-environmental exposures (examples physical fitness)
- Adjust lipid estimates if on statin medication

### Drop variables that don't have weights

Also, some weight variables from the replication are not present, remove them as well


In [5]:
#### REMOVE VARIABLES WITHOUT WEIGHTS
print('Removing from discovery \n')
nhanes.remove_var_without_weights(weights_discovery, phenotypes + covariates)
print('')
print('Removing from replication \n')
nhanes.remove_var_without_weights(weights_replication, phenotypes + covariates)

clarite.describe.summarize(nhanes.data)

Removing from discovery 

Removing URXUAS3 variable from NHANES
Removing LBXWIO variable from NHANES
Removing LBX049 variable from NHANES
Removing LBXBR1 variable from NHANES
Removing spring_water variable from NHANES
Removing LBXV4A variable from NHANES
Removing URXEMM variable from NHANES
Removing URXMMI variable from NHANES
Removing DR1BWATR variable from NHANES
Removing URXUDMA variable from NHANES
Removing URXDCB variable from NHANES
Removing URXUAB variable from NHANES
Removing URXRIM variable from NHANES
Removing LBXBR8 variable from NHANES
Removing URXUP8 variable from NHANES
Removing LBXBR7 variable from NHANES
Removing LBXPFNA variable from NHANES
Removing LBXPFUA variable from NHANES
Removing URXMPB variable from NHANES
Removing LBXEPAH variable from NHANES
Removing URXBUP variable from NHANES
Removing well_water variable from NHANES
Removing DR1TSODI variable from NHANES
Removing DRDSDT6 variable from NHANES
Removing URXPTU variable from NHANES
Removing URXETU variable from

### Drop indeterminate variables and non environmental exposures

In [6]:
# SMQ077 and DDB100 have Refused/Don't Know for "7" and "9" values. We will recode them as nan
nhanes.data = clarite.modify.recode_values(nhanes.data, {7: np.nan, 9: np.nan}, only=['SMQ077', 'DBD100'])

# Droping physical fitness measurements, indeterminate variables and age groups
nhanes.drop_unnecesary_vars(var_category)

nhanes.drop_body_measures(var_category, phenotypes+covariates)
#nhanes.drop_disease_vars(var_category)

clarite.describe.summarize(nhanes.data)

Running recode_values
--------------------------------------------------------------------------------
Replaced 11 values from 18,937 observations in 2 variables
Removing physical fitness variables:
-----------------------------------
Removing variables with indeterminate meanings:
-----------------------------------
Removing age categories:
-----------------------------------
Removing body measures variables:
-----------------------------------
18,937 observations of 1,019 variables
	0 Binary Variables
	0 Categorical Variables
	1,018 Continuous Variables
	1 Unknown-Type Variables



### Adjust lipids if on statins

In [7]:
#Adjust for statins (LDL=LDL/0.7, TC=TC/0.8) (2)
nhanes.adjust_lipids(var_description)

Adjusting lipid variables:
-----------------------------------
	ATORVASTATIN_CALCIUM = ATORVASTATIN_CALCIUM
	SIMVASTATIN = SIMVASTATIN
	PRAVASTATIN_SODIUM = PRAVASTATIN_SODIUM
	FLUVASTATIN_SODIUM = FLUVASTATIN_SODIUM


## Separate discovery and replication, and males and females

In [8]:
split_datasets = nhanes.divide_by_sex()

## Quality control

### Categorize variables

It will also remove all NA variables

In [9]:
for i in range(4):
    split_datasets[i].data = clarite.modify.categorize(split_datasets[i].data)

Running categorize
--------------------------------------------------------------------------------
54 of 1,019 variables (5.30%) are classified as constant (1 unique value).
301 of 1,019 variables (29.54%) are classified as binary (2 unique values).
37 of 1,019 variables (3.63%) are classified as categorical (3 to 6 unique values).
373 of 1,019 variables (36.60%) are classified as continuous (>= 15 unique values).
206 of 1,019 variables (20.22%) were dropped.
	206 variables had zero unique values (all NA).
48 of 1,019 variables (4.71%) were not categorized and need to be set manually.
	47 variables had between 6 and 15 unique values
	1 variables had >= 15 values but couldn't be converted to continuous (numeric) values
Running categorize
--------------------------------------------------------------------------------
63 of 1,019 variables (6.18%) are classified as constant (1 unique value).
274 of 1,019 variables (26.89%) are classified as binary (2 unique values).
40 of 1,019 variable

### Remove constant variables

In [10]:
for l in range(4):
    split_datasets[l].remove_constant_vars(extra_vars=['female', 'male']+covariates+phenotypes)

In [11]:
for i in range(4):
    clarite.describe.summarize(split_datasets[i].data)

4,724 observations of 761 variables
	301 Binary Variables
	37 Categorical Variables
	373 Continuous Variables
	48 Unknown-Type Variables

4,339 observations of 721 variables
	274 Binary Variables
	40 Categorical Variables
	367 Continuous Variables
	38 Unknown-Type Variables

5,123 observations of 787 variables
	290 Binary Variables
	44 Categorical Variables
	407 Continuous Variables
	44 Unknown-Type Variables

4,751 observations of 763 variables
	272 Binary Variables
	42 Categorical Variables
	402 Continuous Variables
	45 Unknown-Type Variables



### Examine unknown variables

All unknown variables are continuous, except area which is categorical, and it's at the end of the lists

In [12]:
for i in range(4):
    var_unknown = split_datasets[i].print_unknown_vars(var_description)
    split_datasets[i].data = clarite.modify.make_continuous(split_datasets[i].data, only=var_unknown[:-1])
    split_datasets[i].data = clarite.modify.make_categorical(split_datasets[i].data, only=var_unknown[-1])

	URXUBE = Beryllium, urine (ng/mL)
	URXUPT = Platinum, urine (ng/mL)
	LBDBANO = Basophils number
	URXPPX = 2-isopropoxyphenol (ug/L)
	quantity_drink_per_day = drink per day
	SMD220 = Age started chewing tobacco regularly
	SMD190 = Age started using snuff regularly
	SMD160 = Age started cigar smoking regularly
	SMD130 = Age started pipe smoking regularly
	DRD350BQ = # of times crabs eaten in past 30 days
	DRD350FQ = # of times oysters eaten in past 30 days
	DRD350HQ = # of times shrimp eaten in past 30 days
	DRD370AQ = # of times breaded fish products eaten
	DRD370DQ = # of times catfish eaten in past 30 days
	DRD370EQ = # of times cod eaten in past 30 days
	DRD370FQ = # of times flatfish eaten past 30 days
	DRD370IQ = # of times perch eaten in past 30 days
	DRD370KQ = # of times pollock eaten in past 30 days
	DRD370MQ = # of times salmon eaten in past 30 days
	DRD370NQ = # of times sardines eaten past 30 days
	DRD370TQ = # of times other fish eaten past 30 days
	DRD370UQ = # of times o

### Clean up variables

- Remove variables that have less than 200 non-NA values
- Remove binary variables that have less than 200 values in a category
- Remove continuous variables with 90% of non-NA observations equal to zero

In [13]:
for i in range(4):
    split_datasets[i].data = clarite.modify.colfilter_min_n(split_datasets[i].data,
                                                            skip=covariates + ['male', 'female'])

Running colfilter_min_n
--------------------------------------------------------------------------------
Testing 296 of 301 binary variables
	Removed 33 (11.15%) tested binary variables which had less than 200 non-null values.
Testing 37 of 38 categorical variables
	Removed 18 (48.65%) tested categorical variables which had less than 200 non-null values.
Testing 418 of 420 continuous variables
	Removed 17 (4.07%) tested continuous variables which had less than 200 non-null values.
Running colfilter_min_n
--------------------------------------------------------------------------------
Testing 269 of 274 binary variables
	Removed 28 (10.41%) tested binary variables which had less than 200 non-null values.
Testing 40 of 41 categorical variables
	Removed 17 (42.50%) tested categorical variables which had less than 200 non-null values.
Testing 402 of 404 continuous variables
	Removed 12 (2.99%) tested continuous variables which had less than 200 non-null values.
Running colfilter_min_n
----

In [14]:
for i in range(4):
    split_datasets[i].data = \
        clarite.modify.colfilter_min_cat_n(split_datasets[i].data,
                       skip=[c for c in covariates + phenotypes + ['male', 'female'] if c in split_datasets[i].data.columns] )

Running colfilter_min_cat_n
--------------------------------------------------------------------------------
Testing 263 of 268 binary variables
	Removed 208 (79.09%) tested binary variables which had a category with less than 200 values.
Testing 19 of 20 categorical variables
	Removed 15 (78.95%) tested categorical variables which had a category with less than 200 values.
Running colfilter_min_cat_n
--------------------------------------------------------------------------------
Testing 241 of 246 binary variables
	Removed 193 (80.08%) tested binary variables which had a category with less than 200 values.
Testing 23 of 24 categorical variables
	Removed 18 (78.26%) tested categorical variables which had a category with less than 200 values.
Running colfilter_min_cat_n
--------------------------------------------------------------------------------
Testing 257 of 262 binary variables
	Removed 200 (77.82%) tested binary variables which had a category with less than 200 values.
Testing 3

In [15]:
# 90percent zero filter
for i in range(4):
    split_datasets[i].data = clarite.modify.colfilter_percent_zero(split_datasets[i].data)

Running colfilter_percent_zero
--------------------------------------------------------------------------------
Testing 403 of 403 continuous variables
	Removed 23 (5.71%) tested continuous variables which were equal to zero in at least 90.00% of non-NA observations.
Running colfilter_percent_zero
--------------------------------------------------------------------------------
Testing 392 of 392 continuous variables
	Removed 20 (5.10%) tested continuous variables which were equal to zero in at least 90.00% of non-NA observations.
Running colfilter_percent_zero
--------------------------------------------------------------------------------
Testing 418 of 418 continuous variables
	Removed 12 (2.87%) tested continuous variables which were equal to zero in at least 90.00% of non-NA observations.
Running colfilter_percent_zero
--------------------------------------------------------------------------------
Testing 423 of 423 continuous variables
	Removed 26 (6.15%) tested continuous variab

### Remove phenotypes that were dropped

Because of the QC, some phenotypes might have been deleted

In [16]:
remove_phenos = []
for i in phenotypes:
    for l in range(4):
        cols = split_datasets[l].data.columns
        if i not in cols:
            print(i + ' was deleted in at least one dataset')
            remove_phenos.append(i)

# Remove the deleted phenos from all datasets
for i in remove_phenos:
    phenotypes.remove(i)
    for l in range(4):
        if i in split_datasets[l].data.columns:
            split_datasets[l].data = split_datasets[l].data.drop(columns=i)

LBXFER was deleted in at least one dataset
LBXEPP was deleted in at least one dataset
LBXTIB was deleted in at least one dataset
LBDPCT was deleted in at least one dataset
LBXIRN was deleted in at least one dataset


In [17]:
for i in range(4):
    clarite.describe.summarize(split_datasets[i].data)

4,724 observations of 442 variables
	60 Binary Variables
	5 Categorical Variables
	375 Continuous Variables
	0 Unknown-Type Variables

4,339 observations of 428 variables
	53 Binary Variables
	6 Categorical Variables
	367 Continuous Variables
	0 Unknown-Type Variables

5,123 observations of 470 variables
	62 Binary Variables
	5 Categorical Variables
	401 Continuous Variables
	0 Unknown-Type Variables

4,751 observations of 463 variables
	58 Binary Variables
	6 Categorical Variables
	397 Continuous Variables
	0 Unknown-Type Variables



In [18]:
# Take note of which variables were differently typed in each dataset
print("Correcting differences in variable types between discovery and replication")
# Merge current type series
dtypes = pd.DataFrame({'discovery_females':clarite.describe.get_types(split_datasets[0].data),
                       'discovery_males':clarite.describe.get_types(split_datasets[1].data),
                       'replication_females':clarite.describe.get_types(split_datasets[2].data),
                       'replication_males':clarite.describe.get_types(split_datasets[3].data),
                       })

for i,l in dtypes.iterrows():
    m = set(l)
    m = {x for x in m if x==x}
    if len(m) > 1:
        print('There is no agreement in here ... ')
        print(i)
        print(l)

# There are no differences

Correcting differences in variable types between discovery and replication


### Keeping variables that are in all datasets

In [19]:
all_vars = set(list(split_datasets[0].data)) & \
           set(list(split_datasets[1].data)) & \
           set(list(split_datasets[2].data)) & \
           set(list(split_datasets[3].data))

for i in range(4):
    split_datasets[i].data = split_datasets[i].data[all_vars]
    
print(f"{len(all_vars)} variables in common")

370 variables in common


## Inspect variables

Inpecting plots before transformations

In [20]:
import warnings
warnings.filterwarnings('ignore')
text = ['_d_f_raw', '_d_m_raw', '_r_f_raw', '_r_m_raw']
plot_paths = os.path.join(paths[2], 'Plots', 'Inspect')
for i in range(4):
    split_datasets[i].plot_variables(plot_paths, phenotypes, covariates, text[i])

Generating a 4 page PDF for 41 variables
Generating a 1 page PDF for 5 variables
Generating a 27 page PDF for 322 variables
Generating a 5 page PDF for 50 variables
Generating a 1 page PDF for 8 variables
Generating a 4 page PDF for 41 variables
Generating a 1 page PDF for 5 variables
Generating a 27 page PDF for 322 variables
Generating a 5 page PDF for 50 variables
Generating a 1 page PDF for 8 variables
Generating a 4 page PDF for 41 variables
Generating a 1 page PDF for 5 variables
Generating a 27 page PDF for 322 variables
Generating a 5 page PDF for 50 variables
Generating a 1 page PDF for 8 variables
Generating a 4 page PDF for 41 variables
Generating a 1 page PDF for 5 variables
Generating a 27 page PDF for 322 variables
Generating a 5 page PDF for 50 variables
Generating a 1 page PDF for 8 variables


## Transform continuous variables

All continuous variables that show greater skew than -0.5 to 0.5. First we will need to merge back the datasets

In [20]:
nhanes = pd.concat([split_datasets[0].data, 
                    split_datasets[1].data, 
                    split_datasets[2].data, 
                    split_datasets[3].data])

nhanes = modules.NhanesData(nhanes)
clarite.describe.summarize(nhanes.data)

18,937 observations of 370 variables
	40 Binary Variables
	5 Categorical Variables
	325 Continuous Variables
	0 Unknown-Type Variables



In [21]:
nhanes.log_transform_skewed()

Tranforming 291 positively skewed, and 6 negatively skewed variables


## Normalize variables

In [22]:
# Merge them to normalize across sexes while keeping the previous variable categories
var_types = clarite.describe.get_types(nhanes.data)
var_continuous  = list(var_types[var_types == 'continuous'].index)
var_binary      = var_types[var_types == 'binary'].index
var_categorical = var_types[var_types == 'categorical'].index
var_constant    = var_types[var_types == 'constant'].index

In [23]:
# Normalize only continuous that are not covariates
for i in covariates:
    if i in var_continuous:
        var_continuous.remove(i)

# Get the columns that are not in the var_continuous list
cols = list(nhanes.data.columns)
for i in var_continuous:
    cols.remove(i)

In [24]:
from sklearn.preprocessing import normalize

nhanes_c = nhanes.data[var_continuous] # nhanes continuous without covariates
nhanes_e = nhanes.data[cols] # nhanes everything else

nhanes_c = (nhanes_c - nhanes_c.min()) / (nhanes_c.max() - nhanes_c.min()) # Normalizing with pandas ignoring NaN

In [25]:
# Merge back
nhanes = clarite.modify.merge_variables(nhanes_e, nhanes_c, how='inner')
nhanes = modules.NhanesData(nhanes)
clarite.describe.summarize(nhanes.data)

Running merge_variables
--------------------------------------------------------------------------------
inner Merge:
	left = 18,937 observations of 48 variables
	right = 18,937 observations of 322 variables
Kept 18,937 observations of 370 variables.
18,937 observations of 370 variables
	40 Binary Variables
	5 Categorical Variables
	325 Continuous Variables
	0 Unknown-Type Variables



## Remove outliers

Only for continuous variables

In [26]:
nhanes.remove_continous_outliers()

h outliers from URXMOP (outside -0.15 to 0.23)
	Removed 0 low and 0 high outliers from DSDCOUNT (outside -0.72 to 1.41)
	Removed 0 low and 21 high outliers from LBXVTO (outside -0.12 to 0.59)
	Removed 0 low and 20 high outliers from LBXF02 (outside -0.07 to 0.67)
	Removed 38 low and 31 high outliers from LBXEOPCT (outside 0.66 to 0.92)
	Removed 15 low and 6 high outliers from LBX187 (outside 0.18 to 0.88)
	Removed 60 low and 25 high outliers from LBXSC3SI (outside 0.19 to 0.74)
	Removed 0 low and 117 high outliers from LBXV4C (outside -0.11 to 0.39)
	Removed 828 low and 0 high outliers from DR1TS040 (outside 0.23 to 1.34)
	Removed 131 low and 121 high outliers from LBXSCR (outside 0.10 to 0.53)
	Removed 116 low and 0 high outliers from DR1TS140 (outside 0.68 to 1.01)
	Removed 0 low and 0 high outliers from SMD057 (outside -0.75 to 1.76)
	Removed 42 low and 23 high outliers from DR1TVK (outside 0.61 to 0.95)
	Removed 35 low and 46 high outliers from LBXWBCSI (outside 0.06 to 0.53)
	Remo

## Inspect transformed data before spliting

In [28]:
plot_paths = os.path.join(paths[2], 'Plots', 'Inspect')
nhanes.plot_variables(plot_paths, phenotypes, covariates, '_total_transformed')

Generating a 4 page PDF for 40 variables
Generating a 1 page PDF for 5 variables
Generating a 28 page PDF for 325 variables
Generating a 5 page PDF for 50 variables
Generating a 1 page PDF for 8 variables


## Spliting dataset back

In [27]:
split_datasets = nhanes.divide_by_sex()

for i in range(4):
    clarite.describe.summarize(split_datasets[i].data)

4,724 observations of 370 variables
	40 Binary Variables
	5 Categorical Variables
	325 Continuous Variables
	0 Unknown-Type Variables

4,339 observations of 370 variables
	40 Binary Variables
	5 Categorical Variables
	325 Continuous Variables
	0 Unknown-Type Variables

5,123 observations of 370 variables
	40 Binary Variables
	5 Categorical Variables
	325 Continuous Variables
	0 Unknown-Type Variables

4,751 observations of 370 variables
	40 Binary Variables
	5 Categorical Variables
	325 Continuous Variables
	0 Unknown-Type Variables



## Inspect tranformed variables split

In [30]:
text = ['_d_f_transformed', '_d_m_transformed', '_r_f_transformed', '_r_m_transformed']
plot_paths = os.path.join(paths[2], 'Plots', 'Inspect')
for i in range(4):
    split_datasets[i].plot_variables(plot_paths, phenotypes, covariates, text[i])

Generating a 4 page PDF for 40 variables
Generating a 1 page PDF for 5 variables
Generating a 28 page PDF for 325 variables
Generating a 5 page PDF for 50 variables
Generating a 1 page PDF for 8 variables
Generating a 4 page PDF for 40 variables
Generating a 1 page PDF for 5 variables
Generating a 28 page PDF for 325 variables
Generating a 5 page PDF for 50 variables
Generating a 1 page PDF for 8 variables
Generating a 4 page PDF for 40 variables
Generating a 1 page PDF for 5 variables
Generating a 28 page PDF for 325 variables
Generating a 5 page PDF for 50 variables
Generating a 1 page PDF for 8 variables
Generating a 4 page PDF for 40 variables
Generating a 1 page PDF for 5 variables
Generating a 28 page PDF for 325 variables
Generating a 5 page PDF for 50 variables
Generating a 1 page PDF for 8 variables


## Remove sex and unnecesary variables

In [28]:
remove_vars = ['male', 'female', 'white'] # The last one is not in the replication weight
for i in range(4):
    split_datasets[i].data = split_datasets[i].data.drop(columns=remove_vars)

In [29]:
for i in range(4):
    clarite.describe.summarize(split_datasets[i].data)

4,724 observations of 367 variables
	39 Binary Variables
	5 Categorical Variables
	323 Continuous Variables
	0 Unknown-Type Variables

4,339 observations of 367 variables
	39 Binary Variables
	5 Categorical Variables
	323 Continuous Variables
	0 Unknown-Type Variables

5,123 observations of 367 variables
	39 Binary Variables
	5 Categorical Variables
	323 Continuous Variables
	0 Unknown-Type Variables

4,751 observations of 367 variables
	39 Binary Variables
	5 Categorical Variables
	323 Continuous Variables
	0 Unknown-Type Variables



## Save cleaned data

In [30]:
os.chdir(paths[2])
savef = ['Discovery_Females', 'Discovery_Males', 'Replication_Females', 'Replication_Males']
for i in range(4):
    split_datasets[i].data.to_csv('CleanData_' + savef[i] + '.csv' )